In [1]:
import numpy as np
import pandas as pd
import sys
import os
from copy import deepcopy
from tqdm.notebook import tqdm
sys.path.append('./utils/')

from DD_data_extractor_git import Data_extractor_v5, normalize, bucketize, split_dataset2, flatten_2D_list, output_vars_v5, split_dataset_multitrain, split_dataset_OddEven

In [ ]:
Data_saveName = 'multitrain_May28'
anatuple_path = "/home/debryas/data/HNL/anatuple/"
period = '2018'
tag = 'AddJETcorr'
channels = ['tee', 'tem', 'tmm', 'tte', 'ttm']

In [3]:
features=[]
features.extend(deepcopy(output_vars_v5))
features.extend(['signal_label', 'channel', 'event_type', 'mass_hyp'])
flat_features = flatten_2D_list(features)
print(f'Number of inputs: {len(flat_features)}')
print(flat_features)

Number of inputs: 135
['event', 'genWeight', 'charge_1', 'charge_2', 'charge_3', 'pt_1', 'pt_2', 'pt_3', 'pt_MET', 'eta_1', 'eta_2', 'eta_3', 'mass_1', 'mass_2', 'mass_3', 'phi_1', 'phi_2', 'phi_3', 'phi_MET', 'deltaphi_12', 'deltaphi_13', 'deltaphi_23', 'deltaphi_1MET', 'deltaphi_2MET', 'deltaphi_3MET', 'deltaphi_1(23)', 'deltaphi_2(13)', 'deltaphi_3(12)', 'deltaphi_MET(12)', 'deltaphi_MET(13)', 'deltaphi_MET(23)', 'deltaphi_1(2MET)', 'deltaphi_1(3MET)', 'deltaphi_2(1MET)', 'deltaphi_2(3MET)', 'deltaphi_3(1MET)', 'deltaphi_3(2MET)', 'deltaeta_12', 'deltaeta_13', 'deltaeta_23', 'deltaeta_1(23)', 'deltaeta_2(13)', 'deltaeta_3(12)', 'deltaR_12', 'deltaR_13', 'deltaR_23', 'deltaR_1(23)', 'deltaR_2(13)', 'deltaR_3(12)', 'pt_123', 'mt_12', 'mt_13', 'mt_23', 'mt_1MET', 'mt_2MET', 'mt_3MET', 'mt_1(23)', 'mt_2(13)', 'mt_3(12)', 'mt_MET(12)', 'mt_MET(13)', 'mt_MET(23)', 'mt_1(2MET)', 'mt_1(3MET)', 'mt_2(1MET)', 'mt_2(3MET)', 'mt_3(1MET)', 'mt_3(2MET)', 'mass_12', 'mass_13', 'mass_23', 'mass_123

In [4]:
values = []
for i in range(len(flat_features)):
    values.append([])
data = dict(zip(flat_features, values))


for channel in tqdm(channels, desc='channels'):
    extractor = Data_extractor_v5(channel)
    data_currchannel = extractor(os.path.join(anatuple_path,period, tag) + '/' + channel + "/anatuple/", data=data)
    # print(data_currchannel.keys())
    for key in data.keys():
        data[key].extend(data_currchannel[key])



channels:   0%|          | 0/5 [00:00<?, ?it/s]

In [5]:
data_dict = data
print(data_dict.keys())
print(np.unique(data_dict['signal_label']))
print(np.unique(data_dict['channel']))

dict_keys(['event', 'genWeight', 'charge_1', 'charge_2', 'charge_3', 'pt_1', 'pt_2', 'pt_3', 'pt_MET', 'eta_1', 'eta_2', 'eta_3', 'mass_1', 'mass_2', 'mass_3', 'phi_1', 'phi_2', 'phi_3', 'phi_MET', 'deltaphi_12', 'deltaphi_13', 'deltaphi_23', 'deltaphi_1MET', 'deltaphi_2MET', 'deltaphi_3MET', 'deltaphi_1(23)', 'deltaphi_2(13)', 'deltaphi_3(12)', 'deltaphi_MET(12)', 'deltaphi_MET(13)', 'deltaphi_MET(23)', 'deltaphi_1(2MET)', 'deltaphi_1(3MET)', 'deltaphi_2(1MET)', 'deltaphi_2(3MET)', 'deltaphi_3(1MET)', 'deltaphi_3(2MET)', 'deltaeta_12', 'deltaeta_13', 'deltaeta_23', 'deltaeta_1(23)', 'deltaeta_2(13)', 'deltaeta_3(12)', 'deltaR_12', 'deltaR_13', 'deltaR_23', 'deltaR_1(23)', 'deltaR_2(13)', 'deltaR_3(12)', 'pt_123', 'mt_12', 'mt_13', 'mt_23', 'mt_1MET', 'mt_2MET', 'mt_3MET', 'mt_1(23)', 'mt_2(13)', 'mt_3(12)', 'mt_MET(12)', 'mt_MET(13)', 'mt_MET(23)', 'mt_1(2MET)', 'mt_1(3MET)', 'mt_2(1MET)', 'mt_2(3MET)', 'mt_3(1MET)', 'mt_3(2MET)', 'mass_12', 'mass_13', 'mass_23', 'mass_123', 'Mt_tot',

In [6]:
data = pd.DataFrame(data_dict)
# data = data.rename(columns={"genWeight": "weightOriginal"})
weightNorm = deepcopy(data['genWeight'])
data['weightNorm'] = weightNorm

In [7]:
N = len(data['event'])
data_norm = normalize(pd.DataFrame(data), 'mass_hyp', N, weight_name='weightNorm')
data_norm = normalize(data_norm, 'signal_label', N, weight_name='weightNorm')
data_norm = normalize(data_norm, 'channel', N/5, weight_name='weightNorm')
data_processed, channel_indices = bucketize(data_norm, 'channel')
print(list(data_processed.keys()))
print(channel_indices)


/home/debryas/HNLAnalysis/HNLclassifier/./utils/DD_data_extractor_git.py:861: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  output[key].replace(list(class_names.keys()), list(class_names.values()), inplace=True)


['event', 'genWeight', 'charge_1', 'charge_2', 'charge_3', 'pt_1', 'pt_2', 'pt_3', 'pt_MET', 'eta_1', 'eta_2', 'eta_3', 'mass_1', 'mass_2', 'mass_3', 'phi_1', 'phi_2', 'phi_3', 'phi_MET', 'deltaphi_12', 'deltaphi_13', 'deltaphi_23', 'deltaphi_1MET', 'deltaphi_2MET', 'deltaphi_3MET', 'deltaphi_1(23)', 'deltaphi_2(13)', 'deltaphi_3(12)', 'deltaphi_MET(12)', 'deltaphi_MET(13)', 'deltaphi_MET(23)', 'deltaphi_1(2MET)', 'deltaphi_1(3MET)', 'deltaphi_2(1MET)', 'deltaphi_2(3MET)', 'deltaphi_3(1MET)', 'deltaphi_3(2MET)', 'deltaeta_12', 'deltaeta_13', 'deltaeta_23', 'deltaeta_1(23)', 'deltaeta_2(13)', 'deltaeta_3(12)', 'deltaR_12', 'deltaR_13', 'deltaR_23', 'deltaR_1(23)', 'deltaR_2(13)', 'deltaR_3(12)', 'pt_123', 'mt_12', 'mt_13', 'mt_23', 'mt_1MET', 'mt_2MET', 'mt_3MET', 'mt_1(23)', 'mt_2(13)', 'mt_3(12)', 'mt_MET(12)', 'mt_MET(13)', 'mt_MET(23)', 'mt_1(2MET)', 'mt_1(3MET)', 'mt_2(1MET)', 'mt_2(3MET)', 'mt_3(1MET)', 'mt_3(2MET)', 'mass_12', 'mass_13', 'mass_23', 'mass_123', 'Mt_tot', 'HNL_CM_a

/home/debryas/HNLAnalysis/HNLclassifier/./utils/DD_data_extractor_git.py:861: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  output[key].replace(list(class_names.keys()), list(class_names.values()), inplace=True)


In [8]:
current_dir = os.getcwd()
print(current_dir)

output_dir = os.path.join(current_dir,"saved_files", "extracted_data")
os.makedirs(output_dir, exist_ok=True)
data_processed.to_pickle(output_dir + f"/Data_{tag}_{period}_"+ Data_saveName)


/home/debryas/HNLAnalysis/HNLclassifier


In [9]:
singletrain = False
if singletrain:
    train, val, test = split_dataset2(data_processed)
    pd.to_pickle(train, output_dir + f"/train_{tag}_{period}_" + Data_saveName)
    pd.to_pickle(val  , output_dir + f"/val_{tag}_{period}_"   + Data_saveName)
    pd.to_pickle(test , output_dir + f"/test_{tag}_{period}_"  + Data_saveName)
else:
    train1, train2, val1,val2 = split_dataset_OddEven(data_processed)
    pd.to_pickle(train1, output_dir + f"/train1_{tag}_{period}_" + Data_saveName)
    pd.to_pickle(train2, output_dir + f"/train2_{tag}_{period}_" + Data_saveName)
    pd.to_pickle(val1  , output_dir + f"/val1_{tag}_{period}_"   + Data_saveName)
    pd.to_pickle(val2  , output_dir + f"/val2_{tag}_{period}_"   + Data_saveName)

Total number of events: 1867310
Train1 set (even): 39.98 %
Train2 set (odd): 40.02 %
Validation1 set (even): 9.99 %
Validation2 set (odd): 10.01 %


In [17]:
train, val, test = split_dataset2(data_processed)
pd.to_pickle(train, output_dir + f"/train_{tag}_{period}_" + Data_saveName)
pd.to_pickle(val  , output_dir + f"/val_{tag}_{period}_"   + Data_saveName)
pd.to_pickle(test , output_dir + f"/test_{tag}_{period}_"  + Data_saveName)

Total number of events :  1867310
Train set : 50.00 %
Validation set : 10.00 %
Test set : 40.00 %
